# Iterative AutoMetrics trial run

This notebook runs the iterative contrastive AutoMetrics pipeline on the press-release dataset and plots metric complexity, test accuracy, and metric lifecycle over iterations.

In [1]:
import os
os.environ['OPENAI_API_KEY'] = open('/Users/spangher/.openai-salt-lab-key.txt').read().strip()
import sys
from pathlib import Path

norm_pkg = Path("/Users/spangher/Projects/stanford-research/norm-research/methods/autometrics")
rfi_marker = "rfi-research/regulations-demo/scripts"

# Remove rfi paths
sys.path = [p for p in sys.path if rfi_marker not in p]

# Put norm autometrics package path first
if str(norm_pkg) not in sys.path:
    sys.path.insert(0, str(norm_pkg))

# Clear cached autometrics modules
for k in list(sys.modules):
    if k == "autometrics" or k.startswith("autometrics."):
        del sys.modules[k]

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import dspy
from autometrics.autometrics import Autometrics
from autometrics.dataset.Dataset import Dataset

[Autometrics] No GPU detected - using BM25 + LLMRec pipeline for CPU-optimized performance


In [3]:
data_path = Path("../datasets/press-releases/press_release_modeling_dataset.csv.gz")
split_dir = data_path.with_suffix("")
output_dir = Path("outputs/iterative_autometrics/press_release_trial")

In [4]:
id_column = "id"
text_column = "output"
label_column = "newsworthiness_score"


df = pd.read_csv(data_path)
df = (
    df
      .rename(columns={"press_release_id": id_column, "text": text_column, "judgement": label_column})
      .dropna(subset=[id_column, text_column, label_column])
)

task_description = (
    "Evaluate press-release text for the human judgment label. The model should score "
    "each press release according to the rubric-driven criteria the LLM proposes."
)

# Dataset is used for task description; the iterative path uses on-disk splits for train/eval/test.
dataset = Dataset(
    dataframe=df,
    target_columns=[label_column],
    ignore_columns=[id_column],
    metric_columns=[],
    name="PressReleaseModeling",
    data_id_column=id_column,
    input_column=text_column,
    output_column=text_column,
    reference_columns=[],
    metrics=[],
    task_description=task_description,
)

len(df)

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_46139/118944994.py:6: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


128131

In [5]:
# Configure LLMs (update model names as needed).
generator_llm = dspy.LM(model="gpt-5-mini")
judge_llm = dspy.LM(model="gpt-5-mini")

In [7]:
auto = Autometrics()

results = auto.run(
    dataset=dataset,
    target_measure=label_column,
    generator_llm=generator_llm,
    judge_llm=judge_llm,
    num_iterations=3,
    data_path=str(data_path),
    split_dir=str(split_dir),
    id_column=id_column,
    text_column=text_column,
    label_column=label_column,
    output_dir=str(output_dir),
    k_pairs=5,
    num_metrics=5,
    num_rubrics=5,
    label_batch_size=200,
    verbose=True,
    max_train_set_size=0.1,
    eval_sample_fraction=0.15,
    tqdm_scoring=True,
)

results

[Autometrics][Iterative] Loaded splits: train=102504, eval=12813, test=12814
[Autometrics][Iterative] Downsampled eval to 1921 rows (fraction=0.15, max=None)
[Autometrics][Iterative] Eval split: selection=1344, gating=577
[Autometrics][Iterative] Iteration 0 generated 5 candidates (requested metrics=5, rubrics=5, attempt 1/3)
Initializing BM25 recommender with index path: /Users/spangher/Library/Application Support/autometrics/bm25_all_metrics
BM25 recommender loaded successfully
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Press_Release_Structure_Markers


Mar 02, 2026 10:09:46 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


[Autometrics][Iterative] Dropped candidate (duplicate rubric): Source_Distribution_Channel
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Brand_first_person_Promotional_Entities
[Autometrics][Iterative] Iteration 0 generated 10 candidates (requested metrics=5, rubrics=5, attempt 2/3)
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Boilerplate_Company_Paragraph
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Executive_Quote_Presence
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Distribution_Readiness
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Journalistic_Utility
[Autometrics][Iterative] Dropped candidate (duplicate rubric): Official_Statement_Authenticity
[Autometrics][Iterative] Iteration 0 prepared 7 candidates for selection (dropped_empty=0, dropped_duplicate=8)
[Autometrics][Iterative] Batched scoring for 7 metrics: total=1344 cached_all=0 missing=1344 (eval_selection_full)


Scoring eval_selection_full:   0%|                                                             | 0/1344 [00:00<?, ?it/s]

[Autometrics][Iterative] Batched scoring chunk 0:200 (eval_selection_full)


Scoring eval_selection_full:  15%|██████▉                                        | 200/1344 [1:00:12<4:38:01, 14.58s/it]

[Autometrics][Iterative] Batched scoring chunk 200:400 (eval_selection_full)


Scoring eval_selection_full:  28%|█████████████▎                                 | 381/1344 [1:51:12<4:30:14, 16.84s/it]

KeyboardInterrupt: 

In [ ]:
iterations_path = output_dir / "iterations.jsonl"
metrics_path = output_dir / "metrics.csv"
test_metrics_path = output_dir / "test_metrics.jsonl"

iterations = pd.read_json(iterations_path, lines=True)
metrics = pd.read_csv(metrics_path)
test_metrics = pd.read_json(test_metrics_path, lines=True)

iterations.head()

## 1. Prompt complexity over time

We proxy prompt complexity by the number of characters in each metric rubric.

In [ ]:
metrics = metrics.copy()
metrics["rubric_len"] = metrics["rubric_text"].fillna("").astype(str).str.len()
metrics["born_iteration"] = metrics["born_iteration"].astype(int)

summary = metrics.groupby("born_iteration")["rubric_len"].agg(["mean", "median", "count"]).reset_index()

plt.figure(figsize=(8, 4))
sns.scatterplot(data=metrics, x="born_iteration", y="rubric_len", alpha=0.7)
sns.lineplot(data=summary, x="born_iteration", y="mean", marker="o", label="mean")
sns.lineplot(data=summary, x="born_iteration", y="median", marker="o", label="median")
plt.title("Metric Prompt Complexity Over Time")
plt.xlabel("Iteration Born")
plt.ylabel("Rubric Length (chars)")
plt.legend()
plt.tight_layout()


## 2. Test accuracy over time

In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(data=test_metrics, x="iteration", y="accuracy", marker="o")
plt.title("Test Accuracy Over Iterations")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.tight_layout()


## 3. Metric additions/removals (lifecycle)

In [ ]:
def _parse_list(val):
    try:
        return json.loads(val) if isinstance(val, str) else list(val or [])
    except Exception:
        return []

metrics = metrics.copy()
metrics["active_iters"] = metrics["active_iterations"].apply(_parse_list)
metrics["birth"] = metrics["born_iteration"].astype(int)

last_iter = int(iterations["iteration"].max()) if not iterations.empty else metrics["birth"].max()
metrics["retired"] = metrics["retired_iteration"].fillna(last_iter).astype(int)

metrics_sorted = metrics.sort_values(["birth", "name"]).reset_index(drop=True)

plt.figure(figsize=(10, max(4, 0.4 * len(metrics_sorted))))
for idx, row in metrics_sorted.iterrows():
    plt.hlines(y=row["name"], xmin=row["birth"], xmax=row["retired"], color="tab:blue", linewidth=3)
    plt.scatter([row["birth"]], [row["name"]], color="tab:green", s=40, label="added" if idx == 0 else None)
    if row["retired_iteration"] == row["retired_iteration"]:  # not NaN
        plt.scatter([row["retired"]], [row["name"]], color="tab:red", s=40, label="removed" if idx == 0 else None)

plt.title("Metric Lifecycle Over Iterations")
plt.xlabel("Iteration")
plt.ylabel("Metric")
plt.legend()
plt.tight_layout()


## Generated metrics on disk

This lists all generated metrics for the task and renders their definitions and prompt templates.

In [ ]:
from IPython.display import Markdown, display
from autometrics.util.generated_metrics import (
    list_generated_metrics_for_task,
    render_generated_metrics_report,
)

generated_metrics_dir = Path("notebooks/generated_metrics")

entries = list_generated_metrics_for_task(
    dataset_name=dataset.get_name(),
    target_measure=label_column,
    seed=42,
    generated_metrics_dir=str(generated_metrics_dir),
)

print(f"Found {len(entries)} generated metrics in {generated_metrics_dir}")
if entries:
    report_md = render_generated_metrics_report(entries)
    display(Markdown(report_md))
else:
    print("No generated metrics found for this task/seed.")
